<a href="https://colab.research.google.com/github/Shambhaviadhikari/PythonClass/blob/main/Music_Recommendations_(Fall_2024)37903602%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Instructions

In this assignment, we will explore practical applications of dimensionality reduction within the context of music recommendation systems.






## Dataset Info

The dataset represents audio features for a number of audio files.

  + The audio files were obtained from artists' music videos on YouTube using the `pytube` / `pytubefix` package.
  + The audio features were obtained using the `librosa` package.
  + Each YouTube video / song was split into 30-second chunks called tracks, and each row represents one of these 30-second tracks.
  + Columns ending with "mean" represent mean values, and columns ending with "var" represent variance values
  + Based on audio sampling methods used, the `track_length` is represented in units where 22050 represents one second.



More info: https://github.com/s2t2/ml-music-2023

## Part 1 - Data Loading




Load the provided dataset of audio features.

Questions:

A) Print the **shape** of the dataset. Print the number of columns. Print the number of rows.

B) In a text cell, describe the **structure** of the data - we have a "row per what?"

C) Print the **column names**.


D) Print the **number of unique songs**, as indicated by the `video_id` column (i.e. 206).

E) Print the **number of unique artists**, as indicated by the `artist_name` column (i.e. 24).




In [ ]:
from pandas import read_csv

TRACK_LENGTH = 30
N_MFCC = 13

csv_filepath = (
    "https://github.com/s2t2/ml-music-2023/raw/main/data/youtube/features_v1/"
    f"length_{TRACK_LENGTH}_mfcc_{N_MFCC}_features.csv"
)
df = read_csv(csv_filepath)
df.head()

,artist_name,video_id,audio_filename,track_number,track_length,tempo,chroma_stft_mean,chroma_stft_var,rms_mean,rms_var,...,mfcc_9_mean,mfcc_9_var,mfcc_10_mean,mfcc_10_var,mfcc_11_mean,mfcc_11_var,mfcc_12_mean,mfcc_12_var,mfcc_13_mean,mfcc_13_var
0,frank_sinatra,rSrc7aulay8,Fly Me To The Moon (In Other Words).mp4,1,661500,123.046875,0.381353,0.108483,0.035173,0.000843,...,6.451812,96.068996,1.410364,101.259422,1.968380,89.808526,9.885193,85.470773,0.186532,85.876381
1,frank_sinatra,rSrc7aulay8,Fly Me To The Moon (In Other Words).mp4,2,661500,117.453835,0.325079,0.095037,0.059724,0.000803,...,2.529402,163.638820,1.359641,111.545326,-0.145113,95.362841,9.985497,107.731330,0.408040,86.850819
2,frank_sinatra,rSrc7aulay8,Fly Me To The Moon (In Other Words).mp4,3,661500,117.453835,0.391230,0.091207,0.062308,0.001372,...,-4.243943,77.650651,5.931900,59.565850,0.710004,56.129559,5.494471,71.811565,-2.069526,62.430867
3,frank_sinatra,rSrc7aulay8,Fly Me To The Moon (In Other Words).mp4,4,661500,117.453835,0.354422,0.093518,0.070922,0.001636,...,-0.558803,111.401847,4.309566,79.829200,1.935659,77.462955,9.476080,86.830481,-2.179213,78.055285
4,frank_sinatra,LWXUdqvVO8Y,Somethin Stupid.mp4,1,661500,103.359375,0.372852,0.090570,0.078486,0.001692,...,-7.024814,67.282919,-6.018267,87.265848,-7.079118,86.895205,-1.009947,69.512053,-9.353167,55.724105


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.decomposition import PCA

In [ ]:
#A) Print the shape of the dataset. Print the number of columns. Print the number of rows.
print(f"The shape of the dataset is {df.shape}. \nIt has {df.shape[0]} rows and {df.shape[1]} columns")


The shape of the dataset is (1818, 46). 
It has 1818 rows and 46 columns


B) In a text cell, describe the structure of the data - we have a "row per what?"

The dataset has 1818 rows and 46 columns. Each row represents a 30 second chunk of the full song. Therefore, we have a row per 30 second chunk per song.

In [ ]:
#C) Print the column names.
print(f"The column names are: {list(df.columns)}")

The column names are: ['artist_name', 'video_id', 'audio_filename', 'track_number', 'track_length', 'tempo', 'chroma_stft_mean', 'chroma_stft_var', 'rms_mean', 'rms_var', 'spectral_centroid_mean', 'spectral_centroid_var', 'spectral_bandwidth_mean', 'spectral_bandwidth_var', 'spectral_rolloff_mean', 'spectral_rolloff_var', 'zero_crossing_rate_mean', 'zero_crossing_rate_var', 'tonnetz_mean', 'tonnetz_var', 'mfcc_1_mean', 'mfcc_1_var', 'mfcc_2_mean', 'mfcc_2_var', 'mfcc_3_mean', 'mfcc_3_var', 'mfcc_4_mean', 'mfcc_4_var', 'mfcc_5_mean', 'mfcc_5_var', 'mfcc_6_mean', 'mfcc_6_var', 'mfcc_7_mean', 'mfcc_7_var', 'mfcc_8_mean', 'mfcc_8_var', 'mfcc_9_mean', 'mfcc_9_var', 'mfcc_10_mean', 'mfcc_10_var', 'mfcc_11_mean', 'mfcc_11_var', 'mfcc_12_mean', 'mfcc_12_var', 'mfcc_13_mean', 'mfcc_13_var']


In [ ]:
#D) Print the number of unique songs, as indicated by the video_id column (i.e. 206).
print(f"There are {df['video_id'].nunique()} songs in the dataset.")

There are 206 songs in the dataset.


In [ ]:
#E) Print the number of unique artists, as indicated by the artist_name column (i.e. 24).
print(f"There are {df['artist_name'].nunique()} artists in the dataset.")

There are 24 artists in the dataset.


## Part 2 - Artist Analysis




Investigate and explore the dataset. Use `pandas` methods. No loops allowed!

A) Rows per Artist:

  + Using `value_counts`, print the number of **rows per artist**.

> NOTE: we should have between 40 and 118 rows per artist.

B) Songs per Artist:

  + Using `groupby`, print the number of unique **songs per artist**, sorted by number of songs in descending order.
  + In a text cell, write the names of the artists who have the greatest number of songs, as well as the name of the artist who has the fewest.
  + Create a horizontal bar chart of the number of songs per artist, where the largest bars are on top. Include chart title and axis labels.

> NOTE: we should have between 6 and 12 unique songs per artist, as represented by the `video_id` .

C) Total Duration in Minutes per Artist:

  + Create a new column called `track_duration_sec` representing the track duration in seconds, and a new column called `track_duration_min` representing the track duration in minutes.
  + Using `groupby`, print the **total duration in minutes per artist**.
  + In a text cell, write the name of the artist who has the longest total duration, and the name of the artist who has the shortest.
  + Create a horizontal bar chart of the total duration in minutes per artist, where the largest bars are on top. Include chart title and axis labels.

> HINT: Divide the length by 22050 to arrive at duration in seconds (see data dictionary), and then again by 60 to arrive at duration in minutes.

> NOTE: we should have between 20 and 59 minutes of audio per artist.

D) Average Tempo per Artist:

  + Using `groupby`, print the **average tempo per artist**.
  + In a text cell, write the name of the artist who has the highest average tempo, and the name of the artist who has the lowest.
  + Create a horizontal bar chart of the average tempo per artist, where the largest bars are on top. Include chart title and axis labels.

E) Your Own Analysis:

  + Continue to explore and ask questions of the data, to better understand it.
  + Write one of your questions in a text cell, then write your own code solution to answer the question.



In [ ]:
#A) Rows per Artist:
#Using value_counts, print the number of rows per artist.
#NOTE: we should have between 40 and 118 rows per artist.

df['artist_name'].value_counts().sort_values()

,count
artist_name,
frank_sinatra,40
ac_dc,49
andrea_bocelli,52
jason_aldean,54
john_legend,54
rihanna,63
alicia_keys,64
led_zeppelin,67
maggie_rogers,68


In [ ]:
#B) Songs per Artist:
# Using groupby, print the number of unique songs per artist, sorted by number of songs in descending order.
unique_artist_songs= df.groupby('artist_name')['video_id'].nunique().sort_values(ascending=False).reset_index()
unique_artist_songs

,artist_name,video_id
0,taylor_swift,12
1,ariana_grande,12
2,chris_stapleton,12
3,coldplay,12
4,jay_z,11
5,john_mayer,10
6,bruce_springsteen,10
7,maggie_rogers,9
8,jason_aldean,8
9,adele,8


In a text cell, write the names of the artists who have the greatest number of songs, as well as the name of the artist who has the fewest.

Artists with the greatest number of songs in the dataset(12 songs):
- Taylor Swift
- Ariana Grande
- Chris Stapleton
- Cold Play

Artist with the fewest number of songs in the dataset(6 songs):
- AC DC

In [ ]:
# Create a horizontal bar chart of the number of songs per artist, where the largest bars are on top. Include chart title and axis labels.
# NOTE: we should have between 6 and 12 unique songs per artist, as represented by the video_id .

barchart_songs = px.bar(unique_artist_songs.sort_values(by= "video_id"), x="video_id", y="artist_name",
                  orientation= "h",
                  title="Bar Chart of the Number of Songs per Artist",
                  labels={
                      'video_id': "Number of Songs",
                      'artist_name' : 'Artist Name'
                  })
barchart_songs.show()

In [ ]:
# C) Total Duration in Minutes per Artist:

# Create a new column called track_duration_sec representing the track duration in seconds, and a new column called track_duration_min representing the track duration in minutes.
df['track_duration_sec'] = df['track_length']/22050
df['track_duration_min'] = df['track_duration_sec']/60
df.head()

,artist_name,video_id,audio_filename,track_number,track_length,tempo,chroma_stft_mean,chroma_stft_var,rms_mean,rms_var,...,mfcc_10_mean,mfcc_10_var,mfcc_11_mean,mfcc_11_var,mfcc_12_mean,mfcc_12_var,mfcc_13_mean,mfcc_13_var,track_duration_sec,track_duration_min
0,frank_sinatra,rSrc7aulay8,Fly Me To The Moon (In Other Words).mp4,1,661500,123.046875,0.381353,0.108483,0.035173,0.000843,...,1.410364,101.259422,1.968380,89.808526,9.885193,85.470773,0.186532,85.876381,30.0,0.5
1,frank_sinatra,rSrc7aulay8,Fly Me To The Moon (In Other Words).mp4,2,661500,117.453835,0.325079,0.095037,0.059724,0.000803,...,1.359641,111.545326,-0.145113,95.362841,9.985497,107.731330,0.408040,86.850819,30.0,0.5
2,frank_sinatra,rSrc7aulay8,Fly Me To The Moon (In Other Words).mp4,3,661500,117.453835,0.391230,0.091207,0.062308,0.001372,...,5.931900,59.565850,0.710004,56.129559,5.494471,71.811565,-2.069526,62.430867,30.0,0.5
3,frank_sinatra,rSrc7aulay8,Fly Me To The Moon (In Other Words).mp4,4,661500,117.453835,0.354422,0.093518,0.070922,0.001636,...,4.309566,79.829200,1.935659,77.462955,9.476080,86.830481,-2.179213,78.055285,30.0,0.5
4,frank_sinatra,LWXUdqvVO8Y,Somethin Stupid.mp4,1,661500,103.359375,0.372852,0.090570,0.078486,0.001692,...,-6.018267,87.265848,-7.079118,86.895205,-1.009947,69.512053,-9.353167,55.724105,30.0,0.5


In [ ]:
# Using groupby, print the total duration in minutes per artist.

artist_dur = df.groupby('artist_name')['track_duration_min'].sum().sort_values(ascending=False).reset_index()
artist_dur

,artist_name,track_duration_min
0,miles_davis,59.0
1,john_coltrane,51.5
2,taylor_swift,50.0
3,coldplay,49.5
4,ariana_grande,46.0
5,chris_stapleton,46.0
6,jay_z,46.0
7,bruce_springsteen,43.5
8,john_mayer,41.5
9,pink_floyd,41.5


In a text cell, write the name of the artist who has the longest total duration, and the name of the artist who has the shortest.

The artist with the longest track time is Miles Davis (59 minutes).
The artist with the the shortest track time is Frank SInatra (20 minutes).

In [ ]:
# Create a horizontal bar chart of the total duration in minutes per artist, where the largest bars are on top. Include chart title and axis labels.
# HINT: Divide the length by 22050 to arrive at duration in seconds (see data dictionary), and then again by 60 to arrive at duration in minutes.
# NOTE: we should have between 20 and 59 minutes of audio per artist.

barchart_dur = px.bar(artist_dur.sort_values(by="track_duration_min"), x='track_duration_min', y='artist_name',
                      orientation='h',
                      title = 'Bar Chart of the Total Duration (minutes) per Artist',
                      labels = {
                          'track_duration_min': 'Track Duration (min)',
                          'artist_name': 'Artist Name'
                      })
barchart_dur.show()

In [ ]:
# D) Average Tempo per Artist:
# Using groupby, print the average tempo per artist.

artist_tempo = df.groupby(['artist_name'])['tempo'].mean().reset_index().sort_values(by='tempo', ascending=False)
artist_tempo

,artist_name,tempo
13,jay_z,131.795568
6,beethoven,129.735525
14,john_coltrane,128.196795
1,adele,127.675907
4,ariana_grande,127.312845
19,miles_davis,126.816801
21,rihanna,126.761981
12,jason_aldean,126.464488
8,chris_stapleton,126.166627
9,coldplay,124.537433


In a text cell, write the name of the artist who has the highest average tempo, and the name of the artist who has the lowest.

The artist with the highest average tempo is Jay Z.
The artist with the lowest everage tempo is Dr Dre.

In [ ]:
# Create a horizontal bar chart of the average tempo per artist, where the largest bars are on top. Include chart title and axis labels.

barchart_tempo = px.bar(artist_tempo.sort_values(by = 'tempo'), x='tempo', y='artist_name',
                        orientation = 'h',
                        title = 'Bar Chart of the Average Tempo per Artist',
                        labels ={
                            'tempo' : 'Average Tempo',
                            'artist_name' : 'Artist Name'
                        })
barchart_tempo.show()

E) Your Own Analysis:
Continue to explore and ask questions of the data, to better understand it.Write one of your questions in a text cell, then write your own code solution to answer the question.

Question:

- Does the data contain any null values?
- What is the duration of each audio file in the dataset? What is the average audio duration?

In [ ]:
#check for the number of missing values in the each feature of the dataset
df.isnull().sum()

,0
artist_name,0
video_id,0
audio_filename,0
track_number,0
track_length,0
tempo,0
chroma_stft_mean,0
chroma_stft_var,0
rms_mean,0
rms_var,0


In [ ]:
#number of tracks per song
audio_tracks = df.groupby(['audio_filename'])['track_duration_sec'].sum().reset_index().sort_values(by='track_duration_sec', ascending=False)
audio_tracks

,audio_filename,track_duration_sec
172,Shine On You Crazy Diamond (Parts I-V).mp4,810.0
152,My Favorite Things.mp4,810.0
55,Blue Train.mp4,630.0
197,Toccata and Fugue in D Minor BWV 565 (Remaster...,540.0
149,Miles Davis - So What (Official Audio).mp4,540.0
...,...,...
90,Do You Know Me.mp4,150.0
45,Ariana Grande - positions (official video).mp4,150.0
89,Crossroads.mp4,120.0
96,Fly Me To The Moon (In Other Words).mp4,120.0


In [ ]:
import time
#average number of tracks per song
track_mean = audio_tracks['track_duration_sec'].mean()
print(f"Average Audio Duration (in minutes): {time.strftime('%M:%S', time.gmtime(track_mean))}")

Average Audio Duration (in minutes): 04:24


## Part 3 - Artist Classification



Perform artist classification using all the provided audio features.

A) X/Y Split:
  + Perform an **X/Y split**. Identify the target variable, and the feature variables.
  + Print the shape of the X and Y datasets, respectively.

B) Data Scaling:

  + **Scale the features** using a standard scaling approach.
  + Print the means and standard deviations for each column in the scaled dataset.

> FYI: all the means should be very small numbers close to zero, while all the standard deviations should be equal to one

C) Train/Test Split:

  + Perform a **train/test split** on the target and scaled features. Use 20% of the rows in the test set. Use a random state for reproducibility.
  + Print the shape of each of the resulting four datasets.

D) Model Training:

  + Using a `LogisticRegression` model from `sklearn`, **train the model** on the training data.
  + Inspect the **coefficients**. Print the shape of the coefficients.
  + In a text cell, write what the shape represents.
  + Display the coefficients as a `pandas.DataFrame`, with appropriate index values and column names.
  + Choose one of the artists, and using the coeficients, determine the top five most predictive features for that artist, and write them in a text cell.

E) Model Evaluation:

  + Evaluate the model using a **classification report**.
  + In a text cell, interpret the classification report results, and describe how well the model did. What percent accuracy represents a "random chance guess" in this particular dataset? Which artists have the highest F1 scores, and which have the lowest?
  + Display a **confusion matrix**, ideally as a heatmap with color.
  + In a text cell, write the names of the top three pairs of artists who were most confused with each other.

### X/Y Split

In [ ]:
#function for returning the X and Y datasets
def xy_split(df, target, drop_var):
    X = df.drop(columns=[*target, *drop_var])
    Y = df[target]
    print(f"X shape: {X.shape}")
    print(f"Y shape: {Y.shape}")
    return X,Y

I decided to remove the track number from the training features because it doesn't provide the model with any valuable information about the track.
I decided to remove track_duration_sec , track_duration_min, and track_length from the training features because all the tracks have the same length. This doesn't add any value to the model.
Finally, I decided the video_id and audio_filename from the training features because they are not numeric features. They are simply identifers for the audio files.

In [ ]:
# A) X/Y Split:
# Perform an X/Y split. Identify the target variable, and the feature variables.
# Print the shape of the X and Y datasets, respectively.

target = ['artist_name']
x, y = xy_split(df, target, ['video_id', 'audio_filename', 'track_number', 'track_length', 'track_duration_sec', 'track_duration_min'])
x.columns

X shape: (1818, 41)
Y shape: (1818, 1)


Index(['tempo', 'chroma_stft_mean', 'chroma_stft_var', 'rms_mean', 'rms_var',
       'spectral_centroid_mean', 'spectral_centroid_var',
       'spectral_bandwidth_mean', 'spectral_bandwidth_var',
       'spectral_rolloff_mean', 'spectral_rolloff_var',
       'zero_crossing_rate_mean', 'zero_crossing_rate_var', 'tonnetz_mean',
       'tonnetz_var', 'mfcc_1_mean', 'mfcc_1_var', 'mfcc_2_mean', 'mfcc_2_var',
       'mfcc_3_mean', 'mfcc_3_var', 'mfcc_4_mean', 'mfcc_4_var', 'mfcc_5_mean',
       'mfcc_5_var', 'mfcc_6_mean', 'mfcc_6_var', 'mfcc_7_mean', 'mfcc_7_var',
       'mfcc_8_mean', 'mfcc_8_var', 'mfcc_9_mean', 'mfcc_9_var',
       'mfcc_10_mean', 'mfcc_10_var', 'mfcc_11_mean', 'mfcc_11_var',
       'mfcc_12_mean', 'mfcc_12_var', 'mfcc_13_mean', 'mfcc_13_var'],
      dtype='object')

### Data Scaling

In [ ]:
#function to scale dataset
def stand_scale(data):
    scaled = ((x - x.mean(axis=0)) / x.std(axis=0))

    return scaled

In [ ]:
#mean and std before standard scaling
x.describe().T[["mean", "std"]]

,mean,std
tempo,1.226104e+02,2.233850e+01
chroma_stft_mean,3.558407e-01,7.029410e-02
chroma_stft_var,8.783250e-02,7.129222e-03
rms_mean,1.470892e-01,8.525706e-02
rms_var,3.976259e-03,4.605240e-03
spectral_centroid_mean,1.946576e+03,5.522269e+02
spectral_centroid_var,4.888274e+05,3.877958e+05
spectral_bandwidth_mean,2.157611e+03,4.321839e+02
spectral_bandwidth_var,2.058261e+05,1.579403e+05
spectral_rolloff_mean,4.038379e+03,1.250180e+03


In [ ]:
# B) Data Scaling:
# Scale the features using a standard scaling approach.
# Print the means and standard deviations for each column in the scaled dataset.
# FYI: all the means should be very small numbers close to zero, while all the standard deviations should be equal to one

x_scaled = stand_scale(x)
x_scaled.describe().T[["mean", "std"]]

,mean,std
tempo,-8.363924e-16,1.0
chroma_stft_mean,-1.563350e-17,1.0
chroma_stft_var,-2.626429e-15,1.0
rms_mean,-1.876020e-16,1.0
rms_var,7.816752e-17,1.0
spectral_centroid_mean,-1.719685e-16,1.0
spectral_centroid_var,7.816752e-17,1.0
spectral_bandwidth_mean,7.738584e-16,1.0
spectral_bandwidth_var,-1.563350e-17,1.0
spectral_rolloff_mean,-1.719685e-16,1.0


### Train Test Split

In [ ]:
# C) Train/Test Split:
# Perform a train/test split on the target and scaled features. Use 20% of the rows in the test set. Use a random state for reproducibility.
# Print the shape of each of the resulting four datasets.

x_train, x_test, y_train, y_test = train_test_split(x_scaled, y,
                                                    test_size=0.2,
                                                    random_state=21)
print("TRAIN:", x_train.shape, y_train.shape)
print("TEST:", x_test.shape, y_test.shape)

TRAIN: (1454, 41) (1454, 1)
TEST: (364, 41) (364, 1)


### Model Training

In [ ]:
# D) Model Training:
# Using a LogisticRegression model from sklearn, train the model on the training data.
model = LogisticRegression(random_state=21)
model.fit(x_train, y_train)

/usr/local/lib/python3.10/dist-packages/sklearn/utils/validation.py:1339: DataConversionWarning:

A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().



LogisticRegression(random_state=21)

In [ ]:
# Inspect the coefficients. Print the shape of the coefficients.
print(f"Shape of Coefficients: {model.coef_.shape}\nCoefficients: \n{model.coef_}")

Shape of Coefficients: (24, 41)
Coefficients: 
[[ 2.35367890e-01  7.99529363e-01 -7.36492179e-01  1.31820758e-01
  -5.26786408e-01  5.39934932e-01  7.67471341e-02  2.48280391e-01
   6.55748899e-01  5.10219017e-01  4.07903761e-01  8.18885446e-01
  -2.31719936e-01  7.03678240e-02 -7.78804522e-01  1.21467678e+00
   1.00655934e-01 -3.31231916e-01 -5.90472799e-02 -1.32816447e+00
   7.79857090e-01 -2.19682754e-01  6.29605986e-01  1.58447064e+00
  -7.02385794e-01  1.33617378e-01  4.29969670e-01 -7.43264403e-01
  -3.43389809e-01  4.51024126e-02  1.83916708e-01 -2.10226378e-01
  -3.80371141e-01 -1.70946234e-01 -6.69635716e-01  1.21726478e+00
  -9.54303037e-02  2.31286749e-01 -2.77532393e-01 -6.29509792e-01
  -2.55789505e-01]
 [ 2.69169921e-01  7.35768971e-02  1.46707119e-01  1.07707913e+00
   4.46275163e-01  3.35334373e-01  9.82280708e-02 -4.45904979e-01
   3.20666732e-01 -1.49387664e-02  4.45233148e-01  1.05818631e+00
  -2.05286255e-01  3.61530862e-02  2.78597909e-01 -2.12580315e+00
  -2.86812

In a text cell, write what the shape represents.

Interpretation of the coefficients:

- The coefficients have 24 rows and 41 columns. For each artist classification, the model has generated a value fo each feature which shows the relationship between each feature and the class.

In [ ]:
# Display the coefficients as a pandas.DataFrame, with appropriate index values and column names.
coefs = pd.DataFrame(model.coef_.T, columns = model.classes_, index=x_train.columns)
coefs

,ac_dc,adele,alicia_keys,andrea_bocelli,ariana_grande,bach,beethoven,bruce_springsteen,chris_stapleton,coldplay,...,john_coltrane,john_legend,john_mayer,led_zeppelin,maggie_rogers,miles_davis,pink_floyd,rihanna,taylor_swift,tupac
tempo,0.235368,0.269170,-0.418251,-0.448590,0.204160,-0.642074,0.136635,-0.336278,0.044814,-0.105187,...,0.429281,-0.252408,-0.140934,-0.198440,-0.291058,0.047677,0.215785,0.384065,-0.251409,0.452769
chroma_stft_mean,0.799529,0.073577,0.555523,-0.810872,1.163807,-0.253832,-1.995847,0.106164,0.155869,-0.698111,...,-1.649764,-0.396537,0.165105,0.489047,1.274888,-2.453913,-0.022016,0.423825,0.524312,1.921656
chroma_stft_var,-0.736492,0.146707,0.512289,-0.206416,0.536918,-1.076809,-1.432876,-0.552779,0.650817,-0.770057,...,1.203003,-0.111666,0.255244,0.329322,0.218501,0.911657,-0.681777,0.944937,-0.308859,0.438822
rms_mean,0.131821,1.077079,1.195231,0.297007,1.842315,-1.079836,-0.534785,-0.018808,0.094512,0.330434,...,-0.840199,-0.126990,1.281006,-1.040807,0.963827,-0.914788,-0.422337,0.017415,-0.007567,-0.238450
rms_var,-0.526786,0.446275,-0.264262,0.359575,0.125060,-0.287041,-0.088122,0.774554,0.070765,-0.732811,...,-0.387448,1.241562,0.043073,-1.243109,0.397829,0.712934,-0.529927,-0.432445,0.246531,-0.162131
spectral_centroid_mean,0.539935,0.335334,-0.676134,-0.537025,0.079079,-0.668999,-0.377316,-0.092612,-0.608774,0.255917,...,-0.517039,0.287924,-0.983025,0.318495,-0.265457,0.843636,-0.509178,0.412188,0.476655,0.365308
spectral_centroid_var,0.076747,0.098228,0.373760,0.443389,-0.058850,-0.384826,-0.324521,0.337144,0.142078,0.546842,...,-0.523926,0.415751,0.135910,-1.486744,-0.632611,-1.032048,-1.038554,-0.313939,0.497675,0.450053
spectral_bandwidth_mean,0.248280,-0.445905,-0.787279,0.110824,0.254976,-1.164878,-0.990179,-1.045851,-1.346463,-0.394102,...,0.860124,-0.132883,0.369495,0.835080,0.361802,0.073011,1.454507,0.234323,0.457520,0.529234
spectral_bandwidth_var,0.655749,0.320667,-1.157269,-0.652057,0.256814,0.017965,-1.559954,-0.347403,-0.197815,0.340416,...,-1.058094,0.508848,-0.182134,-0.677269,-0.269382,-0.080105,0.607950,1.093269,0.848645,0.630822
spectral_rolloff_mean,0.510219,-0.014939,-0.096996,-1.070024,1.020413,-0.318327,-0.604165,-1.021835,0.127455,-0.397718,...,0.127359,0.873995,-1.022833,-0.006142,0.179355,-0.043923,-0.593532,0.197016,-0.253158,0.784645


In [ ]:
# Choose one of the artists, and using the coeficients, determine the top five most predictive features for that artist, and write them in a text cell.
coefs['taylor_swift'].sort_values(ascending = False)[:5]

,taylor_swift
tonnetz_mean,1.475558
mfcc_7_mean,1.264221
mfcc_9_mean,0.980340
mfcc_6_var,0.970360
spectral_bandwidth_var,0.848645


The top five most predictive features for Taylor Swift are mfcc_7_mean (1.684767), tonnetz_mean (1.382950), mfcc_6_var (1.047179), mfcc_9_mean (0.844742), mfcc_2_mean (0.825044).

### Model Evaluation

In [ ]:
from sklearn.metrics import confusion_matrix
import plotly.express as px

def plot_confusion_matrix(y_true, y_pred, height=450, showscale=False, title=None, subtitle=None):
    # https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html
    # Confusion matrix whose i-th row and j-th column
    # ... indicates the number of samples with
    # ... true label being i-th class (ROW)
    # ... and predicted label being j-th class (COLUMN)
    cm = confusion_matrix(y_true, y_pred)

    class_names = sorted(y_test['artist_name'].unique().tolist())

    cm = confusion_matrix(y_test, y_pred, labels=class_names)

    title = title or "Confusion Matrix"
    if subtitle:
        title += f"<br><sup>{subtitle}</sup>"

    fig = px.imshow(cm, x=class_names, y=class_names, height=height,
                    labels={"x": "Predicted", "y": "Actual"},
                    color_continuous_scale="Blues", text_auto=True,
    )
    fig.update_layout(title={'text': title, 'x':0.485, 'xanchor': 'center'})
    fig.update_coloraxes(showscale=showscale)

    fig.show()

In [ ]:
# E) Model Evaluation:
# Evaluate the model using a classification report.

#predict test set
y_pred = model.predict(x_test)
#print classification report
print(classification_report(y_test, y_pred))

                   precision    recall  f1-score   support

            ac_dc       0.92      0.80      0.86        15
            adele       0.74      0.82      0.78        17
      alicia_keys       0.57      0.67      0.62         6
   andrea_bocelli       0.92      1.00      0.96        11
    ariana_grande       0.81      0.87      0.84        15
             bach       0.82      0.78      0.80        18
        beethoven       0.76      0.81      0.79        16
bruce_springsteen       0.75      0.75      0.75        24
  chris_stapleton       0.86      0.86      0.86        14
         coldplay       0.61      0.67      0.64        21
           dr_dre       0.85      0.85      0.85        13
    frank_sinatra       0.67      0.80      0.73         5
     jason_aldean       0.57      0.44      0.50         9
            jay_z       0.75      0.67      0.71        18
    john_coltrane       0.90      0.86      0.88        22
      john_legend       0.64      0.69      0.67       

In a text cell, interpret the classification report results, and describe how well the model did. What percent accuracy represents a "random chance guess" in this particular dataset? Which artists have the highest F1 scores, and which have the lowest?

Classification Report Interpretation:

- Accuracy - the model has an accuracy of 0.75. This indicates that the model did well as it predicted the correct artist for 75% of the songs in the test set correctly.
- Approximately 6.49% represents the random chance guess for this dataset since its an imbalanced dataset. This suggests that if the model were to make random guesses for the artist of each song without learning anything about the patterns and relationships in the data, it would correctly predict the artist for approximately 6.49% of the songs. There is a large difference between the accuracy of the model and the random chance guess. This indicates that the model has done a good job of capturing the patterns in the data and making predictions.
- Ac_Dc, Dr Dre and Bach have the highest f1 scores of 1, 0.92 and 0.92 respectively. This means that for ac_dc the model is doing a perfect job at correctly classifiying songs with their actual artists and it also does a perfect job at not misclassifying ac_dc songs as other artists'. The model also does a very good job at this for the classification of Dr Dre and Bach.
- John Legend, Led Zeppelin and Frank Sinatra have the lowest f1 scores of 0.44, 0.52 and 0.56 respectively. This means that the model doesn't do a great job and distinguishing between their songs and others well enough to correctly classify their songs. Additionally, the model doesn't do a great job at not misclassfying their songs with the songs of other artists.

In [ ]:
# Display a confusion matrix, ideally as a heatmap with color.
plot_confusion_matrix(y_test, y_pred, height=900)

In a text cell, write the names of the top three pairs of artists who were most confused with each other.

The three pairs of artists who were most confued with each other were:

1. Frank Cinatra and Cold Play - the model inaccurately predicted 4 Frank Cinatra songs as Cold Play songs.
2. John Mayer and Chris Stapleton - the model incorrectly predicted 3 John Mayer songs as Chris Stapleton songs.
3. Miles Davis and John Coltrane - the model incaccurately predicted 3 John Coltrane songs as Miles Davis songs.

## Part 4 - Dimensionality Reduction

Perform dimensionality reduction on the audio features.


A) Use a `PCA` model from `sklearn` with two components. Use a random state for reproducibility. **Train the model** on the scaled data (all rows) to obtain the embeddings.

B) Print the explained variance ratio for each component. Also print the **sum of explained variance** for all components. In a text cell, interpret these results.

C) Wrap the **embeddings** in a `pandas.DataFrame`, using appropriate column names ("component_1" and "component_2"), and appropriate index values.

D) **Plot the embeddings** using a scatterplot, with "component_1" on the x axis and "component_2" on the y axis. Color on artist label. In a text cell, interpret the plot.

E) **Calculate the centroids** for each artist. The centroid for a given artist is comprised of the "component_1" mean and the "component_2" mean for that artist.

F) **Plot the centroids** using a scatterplot, with "component_1" on the x axis and "component_2" on the y axis. Color on artist label. In a text cell, interpret the plot. Are there any artists who are in similar areas of the graph? Are there any artists who are more separable than others?

In [ ]:
# A) Use a PCA model from sklearn with two components. Use a random state for reproducibility. Train the model on the scaled data (all rows) to obtain the embeddings.
pca = PCA(n_components=2, random_state=21)

embeddings = pca.fit_transform(x_scaled)
print(f"Shape of the Embeddings: {embeddings.shape}")

Shape of the Embeddings: (1818, 2)


In [ ]:
# B) Print the explained variance ratio for each component. Also print the sum of explained variance for all components. In a text cell, interpret these results.
exp_var_ratio = pca.explained_variance_ratio_
exp_var_sum = sum(pca.explained_variance_)
sum_exp_ratio = 0

for i, j in enumerate(exp_var_ratio):
    sum_exp_ratio += j
    print(f"Principal Component {i+1}")
    print(f"    Explained Variance: {j}")
    print(f"    Cummulative Explained Variance: {sum_exp_ratio}")

print(f"\nSum of Explained Variance: {exp_var_sum}")

Principal Component 1
    Explained Variance: 0.21890505388738138
    Cummulative Explained Variance: 0.21890505388738138
Principal Component 2
    Explained Variance: 0.17100147326771664
    Cummulative Explained Variance: 0.389906527155098

Sum of Explained Variance: 15.986167613359019


Explained Variance Ratio and Explained Variance Interpretations:

- Principal component 1 has an explained variance ratio of approximately 0.2189. This means that approx. 21.89% of the total variance in the dataset is captured by the first principal component.
- Principal component 2 has an explained variance ratio of approximately 0.1710. This means that approx 17.10% of the total variance in the dataset is being captured by the second principal component.
- Together, PC1 and PC2 explains approx. 40% of the total variance in the dataset.
- The sum of explained variance from the results of the PCA is 15.99 which means that the first two components captures variance of 15.98 of the the dataset.

### Track Embeddings Plot

In [ ]:
# C) Wrap the embeddings in a pandas.DataFrame, using appropriate column names ("component_1" and "component_2"), and appropriate index values.
component_names = [f"component_{i+1}" for i in range(2)]

embeddings_df = pd.DataFrame(embeddings, columns = component_names, index = y['artist_name']).reset_index()
embeddings_df

,artist_name,component_1,component_2
0,frank_sinatra,2.202601,0.745840
1,frank_sinatra,2.491992,0.020364
2,frank_sinatra,-1.130122,0.444325
3,frank_sinatra,0.230034,0.480557
4,frank_sinatra,-1.256497,-0.141737
...,...,...,...
1813,taylor_swift,-4.049552,4.266147
1814,taylor_swift,-0.139328,1.263042
1815,taylor_swift,-0.180542,2.521628
1816,taylor_swift,-4.761295,4.381796


In [ ]:
# D) Plot the embeddings using a scatterplot, with "component_1" on the x axis and "component_2" on the y axis. Color on artist label. In a text cell, interpret the plot.
embed_scatter = px.scatter(embeddings_df, x='component_1', y='component_2',
                           color='artist_name',
                           title = "Music Dataset Scatter Plot of Embeddings 2 PCA Components",
                           labels = {
                               'component_1' : 'Component 1',
                               'component_2' : 'Component 2',
                               'artist_name' : 'Artist Name'
                           })
embed_scatter.show()

Interpretation of the Scatter Plot of Embeddings:

- Most artists' songs have negative values in PC1.
- Beethoven songs have very low scores in PC2. Most being negative. PC1 could be a good in helping a model identify the artist Beethoven.
- Its difficult to recognize and interprete patterns in this plot as most artists score similarly in PC1 and PC2.

### Artist Centroids Plot

In [ ]:
# E) Calculate the centroids for each artist. The centroid for a given artist is comprised of the "component_1" mean and the "component_2" mean for that artist.

centroids_df = embeddings_df.groupby('artist_name').mean().reset_index()
centroids_df

,artist_name,component_1,component_2
0,ac_dc,-2.846279,2.790291
1,adele,0.067528,-0.529922
2,alicia_keys,0.636443,-0.094915
3,andrea_bocelli,1.414106,-1.846642
4,ariana_grande,4.120164,1.563161
5,bach,-0.225387,-3.477201
6,beethoven,-1.746756,-5.676132
7,bruce_springsteen,-2.765448,0.809615
8,chris_stapleton,-0.184926,-0.057711
9,coldplay,-1.670077,-0.062192


In [ ]:
# F) Plot the centroids using a scatterplot, with "component_1" on the x axis and "component_2" on the y axis. Color on artist label. In a text cell, interpret the plot. Are there any artists who are in similar areas of the graph? Are there any artists who are more separable than others?

cent_scatter = px.scatter(centroids_df, x='component_1', y ='component_2',
                          color='artist_name',
                           title = "Scatter Plot of Centroids",
                           labels = {
                               'component_1' : 'Component 1',
                               'component_2' : 'Component 2',
                               'artist_name' : 'Artist Name'})
cent_scatter.show()

In a text cell, interpret the plot. Are there any artists who are in similar areas of the graph? Are there any artists who are more separable than others?

Interpretation of the Scatter plot of the Centroids:
- Beethoven is quite distinghuishable from the other artist. He has low negative scores (the lowest compared to other artists) for PC2 and also have low negative scores in PC1. This was seem in the plot of the embeddings as well.  
- Bach songs on average scores low in both PC1 and PC2, just as songs from Beethoven. The songs from these artists may be similar and recommending Bach to a Beethoven lover could be a good idea and vice versa.
- Ariana Grande's songs also seem to be quite seperable from he others. The mean of PC1 is very high for Ariana Grande's songs.
- Dr Dre and Rihanna are also quite distinguishable from the others with PC2 mean of their songs being very positively high. It is also high for PC1. Recommending Dr Dre songs to Rihanna lovers could be a good idea as their songs are similar on PC1 and PC2.
- The PC1 and PC2 means of songs by Miles Davis and Alicia Keys are very close. This could mean that their songs are very similar to each other. The same could be the case for Taaylor Swift and John Legend.
- AC DC is also quite distinguishable from the other artists. Their songs scored very high in PC2 and very negatively low in PC1. Bruce Springsteen scored similarly as AC DC on PC1. Their songs could similar to each other for the elements captured in PC1 and PC2.
- The other artists that cluster around zero on PC2 can be assumed to be similar in nature. However, considering that PC1 and PC2 explain only approx. 40% of the total variance in the data, there could be some missing information. Therefore some important information is missing in helping us determine how actually similar these artists and their songs are.

## Part 5 - PCA Tuning

Determine the ideal number of components to use for dimensionality reduction.

A) Loop through a range of numbers of components `n`, starting at 1, and ending at the number of total features. For each number of components:

  + Train a `PCA` model using that number of components.
  + Print the sum of the explained variance that number of components.
  + Store the sum of the explained variance, along with the corresponding number of components, in a variable so we can plot them later. Consider using a list of dictionaries and then converting it into a `pandas.DataFrame`.

B) **Explained Variance Chart**:

  + Plot a line chart of the explained variance on the y axis and the  number of components on the x axis.
  + In a text cell, interpret the chart. How many components explain 80% of the variance in the original data? How many components explain 90% of the variance?

> NOTE: the explained variance increases as the number of components increases, approaching 100% for all of the original features.

C) **Scree Plot**:

  + Plot a line chart of the eigenvalues on the y axis and the number of components on the x axis.
  + In a text cell, interpret the chart. What is the greatest number of components where the eigenvalue is greater than one? Is there an "elbow" bend in the curve?

> NOTE: the eigenvalues are given by the trained PCA model's `explained_variance_` property, for the PCA model using the maximum number of components.

> NOTE: the eigenvalues decrease as the number of components increases.

D) Based on these results, choose an **optimal number of components** to use, and write your answer in a text cell. Describe why you chose that number. Identify any other potential numbers you were considering.


### Explained Variance Plot

In [ ]:
# A) Loop through a range of numbers of components n, starting at 1, and ending at the number of total features. For each number of components:

# Train a PCA model using that number of components.
# Print the sum of the explained variance that number of components.
# Store the sum of the explained variance, along with the corresponding number of components, in a variable so we can plot them later. Consider using a list of dictionaries and then converting it into a pandas.DataFrame.

exp_var = []

for i in range(1, len(x_scaled.columns)+1):
    pca_tuning = PCA(n_components=i, random_state=21)
    embeddings_tun = pca_tuning.fit_transform(x_scaled)
    print(f"Number of components: {i}")
    print(f"    Shape of Embeddings: {embeddings_tun.shape}")
    exp_var.append({'n_components': i, 'cum_explained_variance_ratio': sum(pca_tuning.explained_variance_ratio_)})
    print(f"    Explained Variance: {sum(pca_tuning.explained_variance_).round(4)}")
    print(f"    Explained Variance Ratio: {pca_tuning.explained_variance_ratio_}")

    eigen_vals = pca_tuning.explained_variance_

Number of components: 1
    Shape of Embeddings: (1818, 1)
    Explained Variance: 8.9751
    Explained Variance Ratio: [0.21890505]
Number of components: 2
    Shape of Embeddings: (1818, 2)
    Explained Variance: 15.9862
    Explained Variance Ratio: [0.21890505 0.17100147]
Number of components: 3
    Shape of Embeddings: (1818, 3)
    Explained Variance: 19.9971
    Explained Variance Ratio: [0.21890505 0.17100147 0.09782746]
Number of components: 4
    Shape of Embeddings: (1818, 4)
    Explained Variance: 22.8575
    Explained Variance Ratio: [0.21890505 0.17100147 0.09782746 0.06976635]
Number of components: 5
    Shape of Embeddings: (1818, 5)
    Explained Variance: 24.7156
    Explained Variance Ratio: [0.21890505 0.17100147 0.09782746 0.06976635 0.0453186 ]
Number of components: 6
    Shape of Embeddings: (1818, 6)
    Explained Variance: 26.2895
    Explained Variance Ratio: [0.21890505 0.17100147 0.09782746 0.06976635 0.0453186  0.03838781]
Number of components: 7
    Shap

In [ ]:
explained_var_df = pd.DataFrame(exp_var)
explained_var_df

,n_components,cum_explained_variance_ratio
0,1,0.218905
1,2,0.389907
2,3,0.487734
3,4,0.557500
4,5,0.602819
5,6,0.641207
6,7,0.672967
7,8,0.700303
8,9,0.726849
9,10,0.750727


In [ ]:
# B) Explained Variance Chart:
# Plot a line chart of the explained variance on the y axis and the number of components on the x axis.

line_exp_var = px.line(explained_var_df, x='n_components', y='cum_explained_variance_ratio',
                       title='Line Chart of Cummulative Explained Variance Ratio for Music Dataset',
                       markers = True,
                       labels = {
                           'cum_explained_variance_ratio' : 'Cummulative Explained Variance Ratio',
                           'n_components' : 'Number of Principal Components'
                       })
line_exp_var.show()


In a text cell, interpret the chart. How many components explain 80% of the variance in the original data? How many components explain 90% of the variance?
NOTE: the explained variance increases as the number of components increases, approaching 100% for all of the original features.

Interpretation of Explained Variance Plot:

- The line graph of the cummulative explained variance plot shows the increase in the proportion of explained variance with each increase in PCA components.
- Just as we saw above, 2 components explains approximately 40% of the data, this is very low.
- The line graph shows us that 12 components explains 80% of the variance of the original data. This is incredible considering the original data has 41 features.
- The line graph also shows that if we use 20 components, 90% of the variance from the original data would be explained. Once again, this is very impressive as we are able to understand and make interpretations about the original data by using about half of the original features.


### Scree Plot

In [ ]:
# C) Scree Plot:
# Plot a line chart of the eigenvalues on the y axis and the number of components on the x axis.
line_eigen = px.line(x=list(range(1,len(eigen_vals)+1)), y=eigen_vals,
                       title='PCA Scree Plot for Music Dataset',
                       markers = True,
                       labels = {
                           'y' : 'Eigen Value',
                           'x' : 'Number of Principal Components'
                       })
line_eigen.add_hline(y=1, line_width=1, line_dash="dot", line_color="red", )
line_eigen.show()


In a text cell, interpret the chart. What is the greatest number of components where the eigenvalue is greater than one? Is there an "elbow" bend in the curve?
NOTE: the eigenvalues are given by the trained PCA model's explained_variance_ property, for the PCA model using the maximum number of components.
D) Based on these results, choose an optimal number of components to use, and write your answer in a text cell. Describe why you chose that number. Identify any other potential numbers you were considering.

NOTE: the eigenvalues decrease as the number of components increases.

Interpretation of Scree Plot:
- Based on the Kaiser Rule, the best number of principal components that would accurately represent the original Music dataset would the first 10.
- Based on the Scree Plot for the Music dataset, the elbow of the plot seems to be at 13 components and its seems to level off from there. This suggests that we should keep 12 components from the PCA.
- The Scree Plot also seems to show another elbow at 11 components suggesting that 10 components should be retained. The cumulative explained variance ratio at 10 components is 75%. This is a good proportion of variance. This also agrees with the Kaiser rule which suggests that we only retain the first 10 compoents.

Therefore, considering that the scree plot levels out after 12 components, the eigen value for 12 is close 1, and that the first 12 components explain 80% of the variance in the original data, I believe the optimal number of components is 12.

### Optimal Number of Components

Answer: 12

## Part 6 - Artist Classification (Reduced Feature Embeddings)

Perform dimensionality reduction on the reduced audio feature embeddings.


A) Reduced Embeddings:

  + Using the number of features you identified in Part 5D, train a `PCA` model on the scaled data.
  + Print the sum of explained variance when using this number of components.
  + Store the embeddings in a `pandas.DataFrame`, using appropriate column names and index values. Add / marge the artist name labels back in for charting later.

B) Classification on the Embeddings:

  + Perform a train/test split using the embeddings as the features.
  + Train a `LogisticRegression` model using the embeddings training data.
  + Evaluate the model using a classification report and confusion matrix.
  + In a text cell, interpret the results. How well did the model do? Which artists have the highest F1 scores, and which have the lowest? Which pairs of artists are most confused with eachother?

C) Results Comparison:

  + In a text cell, compare the classification results from Part 3D using all features, against the classification results from Part 5D using the reduced embeddings as features. What does this tell us about the dimensionality reduction process?

In [ ]:
# A) Reduced Embeddings:
# Using the number of features you identified in Part 5D, train a PCA model on the scaled data.
# Print the sum of explained variance when using this number of components.
pca = PCA(n_components=12, random_state=21)

embeddings = pca.fit_transform(x_scaled)
print(f"Shape of the Embeddings: {embeddings.shape}")
print(f"Sum of Explained Variance when 12 components are retained: {sum(pca.explained_variance_)}")
print(f"Sum of the Proportion of Explained Variance when 12 components are retained: {sum(pca.explained_variance_ratio_)}")

Shape of the Embeddings: (1818, 12)
Sum of Explained Variance when 12 components are retained: 32.58614095863338
Sum of the Proportion of Explained Variance when 12 components are retained: 0.7947839258203263


In [ ]:
# Store the embeddings in a pandas.DataFrame, using appropriate column names and index values. Add / marge the artist name labels back in for charting later.
component_names = [f"component_{i+1}" for i in range(12)]

classif_embeddings_df = pd.DataFrame(embeddings, columns = component_names)
classif_embeddings_df

,component_1,component_2,component_3,component_4,component_5,component_6,component_7,component_8,component_9,component_10,component_11,component_12
0,2.202601,0.745840,-3.912757,1.350484,2.441615,-0.167053,0.145839,-0.421139,-1.587666,-0.160821,1.492257,1.568632
1,2.491992,0.020364,-1.951810,0.211439,2.125171,-0.958780,-0.875155,0.249958,-0.861300,-0.310641,0.963366,-0.139616
2,-1.130122,0.444325,-1.284325,0.489278,1.313492,-0.937059,-0.179890,0.581062,-0.619311,0.023836,1.090626,0.573718
3,0.230034,0.480557,-0.921548,0.009588,1.649616,-1.355418,0.037901,-0.057676,-0.644679,-0.294023,1.039422,0.535166
4,-1.256497,-0.141737,0.925641,1.691576,0.014110,1.229480,-1.272437,1.479420,-0.399058,0.562210,3.004974,0.454971
...,...,...,...,...,...,...,...,...,...,...,...,...
1813,-4.049552,4.266147,-1.475179,-2.260594,0.431776,0.936527,1.040009,1.264727,1.558897,-0.110409,-0.333631,-0.732117
1814,-0.139328,1.263042,-1.349360,-1.760777,-0.209011,1.410555,-0.014386,0.530275,2.736480,0.240519,1.178428,1.199285
1815,-0.180542,2.521628,-2.242287,-2.900636,0.171609,2.012492,1.185284,1.773023,-0.184411,-0.312937,0.740261,-0.017616
1816,-4.761295,4.381796,-1.443139,-1.857723,0.155764,1.007041,1.018307,1.331608,1.436497,-0.088256,-0.217267,-0.226243


In [ ]:
# B) Classification on the Embeddings:
# Perform a train/test split using the embeddings as the features.
x_train, x_test, y_train, y_test = train_test_split(classif_embeddings_df, y,
                                                    test_size=0.2,
                                                    random_state=21)
print("TRAIN:", x_train.shape, y_train.shape)
print("TEST:", x_test.shape, y_test.shape)

TRAIN: (1454, 12) (1454, 1)
TEST: (364, 12) (364, 1)


In [ ]:
# Train a LogisticRegression model using the embeddings training data.
model = LogisticRegression(random_state=21)
model.fit(x_train, y_train)

/usr/local/lib/python3.10/dist-packages/sklearn/utils/validation.py:1339: DataConversionWarning:

A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().



LogisticRegression(random_state=21)

In [ ]:
# Evaluate the model using a classification report and confusion matrix.
#predict test set
y_pred = model.predict(x_test)
#print classification report
print(classification_report(y_test, y_pred))

                   precision    recall  f1-score   support

            ac_dc       0.67      0.80      0.73        15
            adele       0.61      0.65      0.63        17
      alicia_keys       0.29      0.33      0.31         6
   andrea_bocelli       0.82      0.82      0.82        11
    ariana_grande       0.71      0.80      0.75        15
             bach       0.59      0.72      0.65        18
        beethoven       0.71      0.75      0.73        16
bruce_springsteen       0.54      0.58      0.56        24
  chris_stapleton       0.67      0.71      0.69        14
         coldplay       0.50      0.43      0.46        21
           dr_dre       0.73      0.62      0.67        13
    frank_sinatra       0.50      0.20      0.29         5
     jason_aldean       0.43      0.33      0.38         9
            jay_z       0.75      0.67      0.71        18
    john_coltrane       0.70      0.86      0.78        22
      john_legend       0.50      0.46      0.48       

In [ ]:
# Display a confusion matrix
plot_confusion_matrix(y_test, y_pred, height=900, title="Confusion Matrix for Clasification of PCA Music Dataset using 12 Components")

In a text cell, interpret the results. How well did the model do? Which artists have the highest F1 scores, and which have the lowest? Which pairs of artists are most confused with eachother?

Classification Report Interpretation:

- Accuracy - the model has an accuracy of 0.60. This indicates that the model did well as it predicted the correct artist for 60% of the songs in the test set correctly.
- This accuracy score is still significantly better than the 6.49% random chance guess. This makes this model still very good despite using less than half of the columns.
- Andrea Bocelli, John Coltrane and Ariana Grande have the highest f1 scores of 0.82, 0.78 and 0.75 respectively. This means that for these artists the model is doing a great job at correctly classifiying songs with their actual artists and it also does a great job at not misclassifying these artists songs as other artists'.
- Rihanna, Frank Sinatra and Alicia Keys have the lowest f1 scores of 0.14, 0.29 and 0.31 respectively. This means that the model doesn't do a great job of distinguishing between their songs and others well enough to correctly classify their songs. Additionally, the model doesn't do a great job at not misclassfying their songs with the songs of other artists.

Confusion Matrix Interpretation:

The three pairs of artists who were most confued with each other were:

1. Bruce Springsteen and Cold Play - the model inaccurately predicted 5 Cold Play songs as Bruce Springsteen songs.
2. Maggie Rogers and Adele - the model incorrectly predicted 4 Maggie Roggers songs as Adele songs.
3. Miles Davis and John Coltrane - the model incaccurately predicted 4 Miles Davis songs as John Coltrane's.

C) Results Comparison:
In a text cell, compare the classification results from Part 3D using all features, against the classification results from Part 5D using the reduced embeddings as features. What does this tell us about the dimensionality reduction process?

Results Comparison:

- The initial classification model for predicting the artist based on a track using all features of the music dataset has an accuracy score of 75%. The reduced embeddings classification model has an accuracy score of 60%. The reduced embeddings model used 12 components compared to the initial model's 41 features and was still able to correctly predict 60% of the artists correctly. Whilst the model did slightly worse by 15%, it used less than half of the original features making dimensinality reduction an impressive and useful tool. It means that PCA truly retain alot of the importance information contained in the original data. The information that it was able to retain in just 12 components was so valuable that the classification model was still able to recognize patterns and relationships in the embeddings and make accurate predictions.
- The f1 scores for the embeddings model decreased compared to the initial model which used all the features. The artists with the highest and lowest f1 scores differed from the initial model.
- The artists with the most confusion also slightly differed in the embeddings model compared to the initial model. However, it is still impressive that with few components the model was still confused by Miles Davis and John Coltrane's music. This could suggest that their music is truly very similar and even when their songs are compared accross 44 features, their songs are still hard to distinguish from each other.


Overall, the model that used dimensionality reduction still performed very well in comparison to the initial model. PCA was able to capture enough information about the entire dataset in just 12 components rather than 41 and still produce good results in a classification model. Very impressive.
